In [2]:
# Standard library imports
# List files, simple math
import glob
import math

# Array and dataframe handling libraries
import numpy as np
import pandas as pd

# Adds loading/processing bar, to keep you informed
from tqdm.auto import tqdm

# NCBI taxonomy toolkit for taxanomic operations
from ete3 import NCBITaxa

# Initialize the NCBI taxonomy handler
# This downloads/loads the NCBI taxonomy database for querying taxonomic information
# such as lineage, taxon names, and rank information
ncbi = NCBITaxa()

In [3]:
"""
Taxonomy processing utilities for NCBI taxonomic IDs.
This module provides functions to convert NCBI taxon IDs into formatted
taxonomic lineage strings with standardized naming conventions.
"""

# ============================================================================
# Names unclassified taxa by the lowest known subname
# Eg. If only phylum is known, species will be automatically rewritten as "..|s__Pseudomonadota_unclassified"
# ============================================================================

def name_unclassified(name):
    """
    Process a taxonomic name string to handle unclassified or unknown taxa.
    
    This function reformats taxonomic names by identifying the highest
    taxonomic level that is properly classified and then prefixing all
    lower levels with the unclassified prefix.
    
    Args:
        name (str): A taxonomic name string with levels separated by '|'
                   (e.g., "d__Bacteria|p__Proteobacteria|...")
    
    Returns:
        str: Reformatted taxonomic string where unclassified levels are
             properly annotated with the appropriate prefix.
    
    Example:
        >>> name_unclassified("d__Bacteria|p__Proteobacteria|g__unclassified_ASV123")
        "d__Bacteria|p__Proteobacteria|g__unclassified_ASV123"
    """
    # Split the taxonomy string and reverse it to process from lowest to highest rank
    name_list = name.split('|')[::-1]
    
    # Find the highest classified rank (working from lowest to highest)
    for ind, name_element in enumerate(name_list):
        # Remove the rank prefix (e.g., "g__" -> actual name)
        name_element = name_element[3:]
        first_letter = name_element[0]
        
        # Check if this taxon is unclassified
        # Criteria: lowercase first letter (suggests genus sp.), contains "Incertae",
        # or contains "ASV" (Amplicon Sequence Variant)
        if (first_letter.islower() and first_letter.isalpha()) or \
           'Incertae' in name_element or \
           'ASV' in name_element:
            pass  # This rank is unclassified, continue to next
        else:
            break  # Found the highest classified rank
    
    # If the lowest rank was classified, return original string unchanged
    if ind == 0:
        return '|'.join(name_list[::-1])
    
    # Store the prefix for unclassified taxa
    prefix = name_element
    
    # Process all ranks below the classified one (reverse order)
    for ind2, name_element in enumerate(name_list):
        if ind2 == ind:
            break  # Stop when we reach the classified rank
        
        # Extract first letter (rank identifier) and remove prefix
        rank_letter = name_element[0]
        name_element = name_element[3:]
        
        # Reconstruct with unclassified prefix
        name_list[ind2] = f"{rank_letter}__{prefix}_{name_element}"
    
    # Reverse back to proper order and join
    new_val = '|'.join(name_list[::-1])
    return new_val


# ============================================================================
# Global Variables
# ============================================================================

# Dictionary to store taxon ID mappings (taxonomic string -> NCBI ID)
# Can be further used to create correspoding .nwk files!
SID = {}
# Dictionary to cache known broken/failed NCBI taxonomy lookups
# NOTE should be pre-poulated manually !!!
# So far, not usefull in practice at all, too rare cases to work with
BROKEN_NCBI_TAXA = {}


# ============================================================================
# Returns full taxanomic name by NCBI ID, see example in the end of the cell
# ============================================================================

def get_full_name(idn):
    """
    Retrieve and format the full taxonomic lineage for a given NCBI taxon ID.
    
    This function queries the NCBI taxonomy database to get the complete
    lineage for a taxon ID, then formats it as a standardized string with
    rank prefixes (d__, k__, p__, etc.).
    
    Args:
        idn (int): NCBI taxonomic ID (e.g., 9606 for Homo sapiens)
    
    Returns:
        str: Formatted taxonomic lineage string with rank prefixes, or
             cached unclassified name if the ID cannot be resolved.
    
    Example:
        >>> get_full_name(9606)
        "d__Eukaryota|k__Metazoa|p__Chordata|c__Mammalia|o__Primates|f__Hominidae|g__Homo|s__sapiens"
    
    Notes:
        - Uses global SID and BROKEN_NCBI_TAXA dictionaries for caching
        - Handles unclassified taxa with the name_unclassified function
        - Returns np.nan if no valid taxonomy can be found
    """
    try:
        # Define the taxonomic ranks in order (domain to species)
        ordered_orgs = ['d', 'k', 'p', 'c', 'o', 'f', 'g', 's']
        
        # Get the lineage translation (taxon names for each ancestor)
        # [1:] removes the root node (taxon ID 1)
        lpath = ncbi.get_lineage_translator([idn])[idn][1:]
        
        # Get the rank for each taxon in the lineage
        # Handle special cases: 'acellular root' -> 'domain', 'clade' -> 'xxx'
        orgs = [
            ncbi.get_rank([ii])[ii].replace('acellular root', 'domain').replace('clade', 'xxx')[0]
            for ii in lpath
        ]
        
        # Get the scientific names for each taxon ID
        lpath_names = [ncbi.get_taxid_translator([i])[i] for i in lpath]
        
        # Create mapping of rank to name, fill missing with 'unclassified'
        d = dict(zip(orgs, lpath_names))
        d = [(o, d.get(o, 'unclassified')) for o in ordered_orgs]
        
        # Format as "rank__name" strings
        lpath_names = ['__'.join(vv) for vv in list(d)]
        
    except Exception as e:
        # Handle any NCBI database errors
        if idn not in BROKEN_NCBI_TAXA.keys():
            # Log the failure (commented out to avoid excessive output)
            # print('NCBI error', idn)
            try:
                # Try to process with the cached unclassified name
                SID[name_unclassified(BROKEN_NCBI_TAXA.get(idn, np.nan))] = idn
            except:
                pass  # Silently fail if even the fallback doesn't work
        return BROKEN_NCBI_TAXA.get(idn, np.nan)
    
    # Successfully retrieved taxonomy - store in SID mapping
    # Replace spaces with underscores for consistent formatting
    formatted_taxonomy = '|'.join(lpath_names).replace(' ', '_')
    SID[name_unclassified(formatted_taxonomy)] = idn
    
    # Return the formatted taxonomy string
    return formatted_taxonomy


# ============================================================================
# Example
# ============================================================================

#Example: Get full taxonomy for E. coli
get_full_name(562)

'd__Bacteria|k__Pseudomonadati|p__Pseudomonadota|c__Gammaproteobacteria|o__Enterobacterales|f__Enterobacteriaceae|g__Escherichia|s__Escherichia_coli'

In [4]:
"""
Microbiome data processing utilities for rarefaction and taxonomic aggregation.

Provides functions for:
1. Subsample/rarefy count data with controlled randomness
2. Perform multiple rarefaction replicates and average results
3. Aggregate taxonomic data at different ranks (derive tables)
"""

# ============================================================================
# Rarefaction / Subsampling Functions
# ============================================================================

def subsample_counts(counts, n, seed=7):
    """
    Subsample (rarefy) a count vector to a specified depth.
    
    This function performs random subsampling without replacement from a
    population defined by the input count vector. It's commonly used in
    microbiome analysis to normalize sequencing depth across samples.
    
    Args:
        counts (np.ndarray): 1D array of counts per category (e.g., ASV counts)
        n (int): Number of reads to subsample (target depth)
        seed (int, optional): Random seed for reproducibility. Defaults to 7.
    
    Returns:
        np.ndarray: Subsampled count vector with same shape as input,
                    where sum(result) == n
    
    Example:
        >>> counts = np.array([10, 20, 30, 40])
        >>> subsample_counts(counts, 50, seed=42)
        array([5, 12, 15, 18])
    
    Notes:
        - Uses NumPy's default random number generator for better performance
        - The total sum of the result equals n (or less if counts don't have enough)
    """
    # Create a random number generator with the specified seed
    rng = np.random.default_rng(seed=seed)
    
    # Expand the counts into individual indices
    # e.g., counts=[2,3] -> unpacked=[0,0,1,1,1]
    unpacked_indices = np.repeat(np.arange(counts.size), counts)
    
    # Randomly select n indices without replacement
    selected_indices = rng.choice(unpacked_indices, size=n, replace=False)
    
    # Count how many times each original index was selected
    unique_indices, frequencies = np.unique(selected_indices, return_counts=True)
    
    # Create result array initialized with zeros
    result = np.zeros_like(counts)
    result[unique_indices] = frequencies
    
    return result


def subsample_counts_with_repeats(data, repeats=999):
    """
    Perform multiple rarefaction replicates and average the results.
    
    This function applies subsampling with multiple random seeds and averages
    the results to reduce stochastic variation in rarefaction. Each row
    (sample) is rarefied to the minimum total depth across all samples.
    
    Args:
        data (pd.DataFrame): Count table with samples as rows and taxa as columns
        repeats (int, optional): Number of rarefaction replicates. Defaults to 999.
    
    Returns:
        pd.DataFrame: Averaged rarefied count table with same dimensions as input
    
    Example:
        >>> df = pd.DataFrame({'ASV1': [10, 5], 'ASV2': [20, 15]}, index=['S1', 'S2'])
        >>> result = subsample_counts_with_repeats(df, repeats=100)
    
    Notes:
        - Uses tqdm for progress visualization
        - The target depth is the minimum row sum (min sequencing depth)
        - Multiple replicates help stabilize the rarefaction process
    """
    # Calculate the minimum sequencing depth across all samples
    min_depth = data.sum(axis=1).min()
    print(f'Rarefying to minimum depth: {min_depth} reads')
    
    # List to store processed rows
    processed_rows = []
    
    # Process each sample (row) with a progress bar
    for row in tqdm(data.values, desc="Rarefying samples", unit="sample"):
        # Convert row to list and then to numpy array
        row_counts = np.array(row.tolist())
        
        # Perform multiple rarefaction replicates with different seeds
        # Each replicate uses a seed from 0 to repeats-1
        replicate_results = [
            subsample_counts(row_counts, min_depth, seed=seed)
            for seed in range(repeats)
        ]
        
        # Average the results across all replicates
        averaged_row = np.array(replicate_results).mean(axis=0)
        processed_rows.append(averaged_row)
    
    # Create a new DataFrame with averaged results
    processed_data = pd.DataFrame(
        processed_rows,
        index=data.index,
        columns=data.columns
    )
    
    return processed_data


# ============================================================================
# Taxonomic Aggregation to higher levels, TSS
# ============================================================================

def derive_tables(data):
    """
    Generate a series of taxonomic tables at different rank levels.
    
    This function takes a count table with full taxonomic lineage strings
    and creates aggregated versions by progressively removing the lowest
    taxonomic rank. The result is a list of tables from finest to coarsest
    resolution.
    
    Args:
        data (pd.DataFrame): Count table with samples as rows and taxa as columns.
                            Column names should be taxonomic strings with
                            ranks separated by '|' (e.g., "d__Bacteria|p__...|s__...")
    
    Returns:
        list: List of pandas DataFrames, where index 0 is the original table,
              index 1 is aggregated at one rank higher, index 2 at two ranks
              higher, etc., up to the highest rank (domain).
    
    Notes:
        - Creates deep copies of data to avoid modifying the original
        - Aggregates by summing counts at each taxonomic rank
        - Uses groupby operations for aggregation
        - Note: In rare mishandling cases spits duplicated columns, check manually in the end of processing
    """
    # List to store the derived tables
    derived_tables = []
    
    # Create a deep copy of the original data to preserve it
    data = data.copy(deep=True)
    data.columns.name = 'TAXA'  # Name the columns for easier manipulation
    
    # Add the original table as the first element (finest resolution)
    derived_tables.append(data.copy(deep=True))
    
    # Reset the index to work with taxonomy as a column
    # This makes it easier to manipulate the taxonomic strings
    data = data.T.reset_index()
    
    # Determine the number of taxonomic levels present
    # e.g., "d__Bacteria|p__..." -> count of '|' + 1
    depth = len(data['TAXA'].values[0].split('|'))
    
    # Iteratively remove the lowest taxonomic level and aggregate
    # We do this for each rank level (from species up to domain)
    for _ in range(depth - 1):
        # Remove the last taxonomic level (lowest rank)
        data['TAXA'] = data['TAXA'].apply(lambda x: '|'.join(x.split('|')[:-1]))
        
        # Aggregate counts by the higher-level taxonomy
        # Sum counts for all taxa that share the same higher-level classification
        data = data.groupby('TAXA').sum()
        
        # Transpose back so samples are rows and taxa are columns
        data = data.T
        
        # Store this aggregated table
        derived_tables.append(data.copy(deep=True))
        
        # Transpose again and reset index for the next iteration
        # This prepares the data for the next level of aggregation
        data = data.T.reset_index()
    
    # Return the list of tables from finest to coarsest resolution
    return derived_tables

In [5]:
"""
Kraken2 report loading and filtering configuration
"""

# ============================================================================
# Kraken Report File Loading
# ============================================================================

# List all Kraken2 report files
# These are tab-separated files containing taxonomic classification results
# from Kraken2, with columns for: percentage, reads, taxon counts, rank, taxid, name
FILES = glob.glob('example_data/*.txt')

# ============================================================================
# Taxonomic Filtering (if required)
# ============================================================================

# List of viral families to retain for sensitive Kraken2 analysis
# This filter serves two purposes:
# 1. Remove potential contamination (non-target organisms)
# 2. Focus analysis on specific families of interest
LFAMS = [] # no filtration

LFAMS = [
    'Caudoviricetes_unclassified',
    'Anelloviridae',
    'Intestiviridae',
    'Aliceevansviridae',
    'Suoliviridae',
    'Paramyxoviridae',
    'Peduoviridae',
    'Matonaviridae',
    'Casjensviridae',
    'Papillomaviridae',
    'Orthoherpesviridae',
    'Rountreeviridae'
]

In [6]:
"""
Kraken2 report processor for metagenomics analysis with sensitive settings

This module processes Kraken2 classification reports to:
1. Reassing taxa name using current NCBI taxonomy
2. Focus on specific families of interest (if required)
3. Limit read count of species level by total read count on more confident genus level
"""

# ============================================================================
# Data Containers, Diagnostics
# ============================================================================

# Container for problematic/unclassified entries (for diagnostic purposes)
NA_ENTRIES = []
# Container for processed results (final dataframes)
RES = []

# ============================================================================
# Main Processing Loop
# ============================================================================

# Process each Kraken2 report file with a progress bar
for FILE in tqdm(FILES, desc="Processing Kraken reports", unit="file"):

    # ========================================================================
    # Step 1: Extract Sample ID and Load Data
    # ========================================================================

    # Extract sample ID from filename
    # Correct accordingly
    SAMPLE_ID = FILE.split('/')[-1].split('_')[0]

    # Load Kraken2 report file
    # Format: %readcount, total_reads, reads_at_rank, rank_code, taxid, taxon_name
    df = pd.read_table(FILE, header=None)
    df.columns = ['prcnt', 'reads_clade', 'reads', 'org', 'id', 'name']

    # ========================================================================
    # Step 2: Map TaxIDs to Full Taxonomic Lineage
    # ========================================================================

    # Clean up taxonomic names (remove trailing whitespace)
    df['name'] = df['name'].str.strip()

    # Get full taxonomic lineage for each taxon ID
    df['map_name'] = df['id'].apply(lambda x: get_full_name(x))

    # Store entries that couldn't be mapped (diagnostic)
    NA_ENTRIES.append(df[df.map_name.isna()])

    # Remove unmapped entries
    # In practice fail NCBI records are extra rare taxa (so far)
    df = df.dropna(subset=['map_name'])

    # Extract domain and family from the taxonomic lineage
    df['domain'] = df['map_name'].apply(lambda x: x.split('|')[0])
    # Family is the third from last level
    df['family'] = df['map_name'].apply(lambda x: x.split('|')[-3][3:])

    # ========================================================================
    # Step 3: Filter (if required)
    # ========================================================================

    # Keep only viruses (based on NCBI taxonomy domain)
    df = df[df.domain == 'd__Viruses']

    # Store number of reads assigned to genus level
    # On genus level assignment is more cofident than on species level
    CONFIDENT = df[df.org.isin(['G', 'G1'])]

    # Keep only species-level classifications (S = species)
    # Optional: Could include 'S1', but be aware in some domains it will lead to Species and Strain mixing
    # Possible solution: process separetly, then detatch and combine
    df = df[df.org.isin(['S'])]

    # Optional depth filter
    # Very sound option, yet may be too much for analysis of low-count domain, such as viruses
    # df = df[df.reads > 50]

    # Calculate total species-level reads for this sample
    # To rescale to this approx sample depth later
    SPECIES_READS = df.reads.sum()

    # ========================================================================
    # Step 4: Filter for target families (if required)
    # ========================================================================

    # Keep only taxa from specified viral families (LFAMS)
    if LFAMS:
        df = df[df.family.isin(LFAMS)]
        print(f"Sample {SAMPLE_ID} dims: {df.shape[0]}")

    # Skip empty samples
    if df.empty:
        continue

    # Remove human contamination (if present), if domain is not prefiltered
    df = df[~df.map_name.str.contains('Hominidae')]

    # ========================================================================
    # Step 5: Process genus levels more confident read number
    # ========================================================================

    # Extract genus-level classification for confident assignments
    # Second from last level (format: g__GenusName)
    df['CONFIDENT'] = df['map_name'].apply(lambda x: x.split('|')[-2][3:])

    # Process confident classifications separately
    CONFIDENT['name'] = CONFIDENT['name'].str.strip()
    CONFIDENT['name2'] = CONFIDENT['id'].apply(get_full_name)

    # Remove any null entries
    CONFIDENT = CONFIDENT[~CONFIDENT.name2.isna()]

    # Extract genus-level names from confident classifications
    CONFIDENT['name2'] = CONFIDENT['name2'].apply(
        lambda x: x.split('|')[-2][3:])

    # Create a mapping of genus level total reads
    CONFIDENT_READS_MAP = CONFIDENT.groupby('name2').reads.sum().to_dict()

    # Add confident read counts (i.e. reads assigned at genus level) to the main dataframe
    df['CONFIDENT_reads'] = df['CONFIDENT'].map(CONFIDENT_READS_MAP).values

    # ========================================================================
    # Step 6: Handle missing data
    # ========================================================================

    # Replace zeros with NaN for better handling
    df['CONFIDENT_reads'] = df['CONFIDENT_reads'].fillna(0)
    df['CONFIDENT_reads'] = df['CONFIDENT_reads'].replace(0, np.nan)
    df['reads'] = df['reads'].replace(0, np.nan)

    # Use clade-level reads as fallback for missing species reads
    # Clade reads include all reads assigned to this taxon's clade
    df['reads'] = df['reads'].fillna(df['reads_clade'])
    df['CONFIDENT_reads'] = df['CONFIDENT_reads'].fillna(df['reads'])

    # Remove any remaining NaN values
    df = df.dropna(subset=['reads', 'reads_clade'])

    # ========================================================================
    # Step 7: Weighting scheme for higher confidence assignment
    # ========================================================================

    # Process each confident genus separately
    confident = []

    for confident_genus in df.CONFIDENT.unique():

        # Get subset for this confident genus
        sub_df = df[df.CONFIDENT == confident_genus]

        # Apply weighting to reduce impact of low-confidence uniformly distributed assignments
        # Square the reads to favor more abundant taxa
        sub_df['reads'] = sub_df['reads']**2

        # reads to %
        sub_df['PCNT'] = sub_df['reads'] / (sub_df['reads'].sum() + 1)

        # Scale % of reads by species in genus by total reads assigned to
        # more confident higher level of classification
        sub_df[
            'CONFIDENT_reads_by'] = sub_df['CONFIDENT_reads'] * sub_df['PCNT']

        # Notify if reads sum is zero
        if sub_df['reads'].sum() == 0:
            print(f'Warning: Zero reads for confident genus {confident_genus}')
            sub_df['CONFIDENT_reads_by'] = 0

        confident.append(sub_df)

    # Combine all processed subsets
    df = pd.concat(confident)

    # ========================================================================
    # Step 8: Final count calculation
    # ========================================================================

    # Clean up the weighted reads
    df['CONFIDENT_reads_by'] = df['CONFIDENT_reads_by'].fillna(0)

    # Round up to ensure integer read counts
    df['CONFIDENT_reads_by'] = df['CONFIDENT_reads_by'].apply(
        lambda x: math.ceil(x))

    # Replace original reads with weighted confident reads
    df['reads'] = df['CONFIDENT_reads_by'].values

    # Scale back to original species-level read total
    # This preserves the total reads while redistributing based on confidence
    df['reads'] = df['reads'] / df['reads'].sum() * SPECIES_READS

    # ========================================================================
    # Step 9: Format and Store Results
    # ========================================================================

    # Sort by read count (descending)
    # Rows = samples, Columns = taxa, Values = read counts
    df = df.sort_values(by='reads',
                        ascending=False).set_index('map_name')[['reads']].T
    df.index = [SAMPLE_ID]

    # Append to results list
    RES.append(df)

Processing Kraken reports:  25%|███████████▌                                  | 1/4 [00:02<00:06,  2.04s/file]

Sample HPV401 dims: 13


Processing Kraken reports:  75%|██████████████████████████████████▌           | 3/4 [00:02<00:00,  1.70file/s]

Sample HPV402 dims: 3
Sample HPV403 dims: 3


Processing Kraken reports: 100%|██████████████████████████████████████████████| 4/4 [00:02<00:00,  1.54file/s]

Sample HPV404 dims: 4


In [7]:
# Concant sample level processing to one dataframe
RES = pd.concat(RES).fillna(0).astype('int')

In [8]:
# Derive hierarchy and TSS relative abundance
datas = derive_tables(RES)
COUNTS = pd.concat(datas[::-1], axis=1).T.reset_index().groupby('TAXA').sum().T
RELAB = pd.concat([d.div(d.sum(axis=1), axis=0) * 100 for d in datas[::-1]], axis=1)

COUNTS.columns = [name_unclassified(c) for c in COUNTS.columns]
RELAB.columns = [name_unclassified(c) for c in RELAB.columns]

In [11]:
c = RELAB.filter(regex=f'f__[A-z0-9-.]+$').max().sort_values(ascending=False)
c = c[c>0]
c

d__Viruses|k__Heunggongvirae|p__Uroviricota|c__Caudoviricetes|o__unclassified|f__Rountreeviridae          62.500000
d__Viruses|k__Heunggongvirae|p__Uroviricota|c__Caudoviricetes|o__unclassified|f__Peduoviridae             50.602410
d__Viruses|k__Shotokuvirae|p__Cossaviricota|c__Papovaviricetes|o__Zurhausenvirales|f__Papillomaviridae    50.340136
d__Viruses|k__Orthornavirae|p__Negarnaviricota|c__Monjiviricetes|o__Mononegavirales|f__Paramyxoviridae    25.000000
d__Viruses|k__Heunggongvirae|p__Uroviricota|c__Caudoviricetes|o__unclassified|f__Casjensviridae            6.024096
dtype: float64

In [12]:
RELAB[c.index]

,d__Viruses|k__Heunggongvirae|p__Uroviricota|c__Caudoviricetes|o__unclassified|f__Rountreeviridae,d__Viruses|k__Heunggongvirae|p__Uroviricota|c__Caudoviricetes|o__unclassified|f__Peduoviridae,d__Viruses|k__Shotokuvirae|p__Cossaviricota|c__Papovaviricetes|o__Zurhausenvirales|f__Papillomaviridae,d__Viruses|k__Orthornavirae|p__Negarnaviricota|c__Monjiviricetes|o__Mononegavirales|f__Paramyxoviridae,d__Viruses|k__Heunggongvirae|p__Uroviricota|c__Caudoviricetes|o__unclassified|f__Casjensviridae
HPV401,62.500000,37.500000,0.000000,0.0,0.000000
HPV402,43.373494,50.602410,0.000000,0.0,6.024096
HPV403,25.000000,50.000000,0.000000,25.0,0.000000
HPV404,24.489796,25.170068,50.340136,0.0,0.000000
